<a href="https://colab.research.google.com/github/GayoungJeong/customer-segmentation/blob/main/03_RFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
clean = pd.read_csv('/content/drive/MyDrive/marketing-portfolio/data/retail_clean.csv')
print(len(clean))   # 824364

Mounted at /content/drive
824364


In [2]:
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'])

In [3]:
purchases = clean[~clean['Invoice'].astype(str).str.startswith('C')]
print(len(purchases))  #R,F는 실제로 산 기록만 봐야하므로 환불건 삭제

805620


In [4]:
clean.groupby('Customer ID')['TotalPrice'].sum()

,TotalPrice
Customer ID,
12346.0,-64.68
12347.0,5633.32
12348.0,2019.40
12349.0,4404.54
12350.0,334.40
...,...
18283.0,2736.65
18284.0,436.68
18285.0,427.00


In [5]:
ref_date = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(ref_date)

2011-12-10 12:50:00


In [6]:
last_purchase = purchases.groupby('Customer ID')['InvoiceDate'].max()   # 가장 늦은 날짜 = 최댓값
recency = (ref_date - last_purchase).dt.days                            # 며칠 차이인지

In [7]:
frequency = purchases.groupby('Customer ID')['Invoice'].nunique() #중복을 빼고 개수가 몇 개인지

In [8]:
monetary = clean.groupby('Customer ID')['TotalPrice'].sum()

In [9]:
rfm = pd.DataFrame({'Recency': recency, 'Frequency': frequency, 'Monetary': monetary})
print(len(rfm))   # 고객 수
rfm.head()

5942


,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326.0,12.0,-64.68
12347.0,2.0,8.0,5633.32
12348.0,75.0,5.0,2019.40
12349.0,19.0,4.0,4404.54
12350.0,310.0,1.0,334.40


In [10]:
rfm.describe()

,Recency,Frequency,Monetary
count,5881.000000,5881.000000,5942.000000
mean,201.457745,6.287196,2801.799459
std,209.474135,13.012879,13973.922553
min,1.000000,1.000000,-25111.090000
25%,26.000000,1.000000,325.097500
50%,96.000000,3.000000,843.970000
75%,380.000000,7.000000,2182.005000
max,739.000000,398.000000,598215.220000


R,F에는 취소건만 있는 고객이 빠짐. Monetary에는 총 고객수가 들어가 있음. 따라서 5942-5881= 61명은 취소건만 있는 고객이고 Monetary에서 추가로 뺄 항목은 머니터리 항목이 0이하인 고객. (샀다가 전부(또는 그 이상) 취소한 고객은 회사에 남긴 매출이 없어서, 가치 점수를 매기는 대상이 아니에요. 다만 **”전량 반품 고객"**은 따로 볼 만한 그룹이에요. 왜 반품했는지 알아보는 건 다른 마케팅 과제) 그래서 그 수가 41명이고, 총 102명 제외(머니터리가 0이하인 수)=총 5840명

In [11]:
low = rfm[rfm['Monetary'] <= 0]   # Monetary가 0 이하인 고객만
print(len(low))                    # 몇 명인지
low.head()

102


,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326.0,12.0,-64.68
12382.0,NaN,NaN,-18.38
12467.0,423.0,1.0,0.00
12503.0,584.0,1.0,0.00
12706.0,NaN,NaN,-91.89


In [12]:
rfm.loc[16446]

,16446.0
Recency,1.0
Frequency,2.0
Monetary,-6.1


In [13]:
has_purchase = rfm[rfm['Recency'].notnull()]          # 구매 기록 있는 고객만
print(len(has_purchase[has_purchase['Monetary'] <= 0]))  # 그중 Monetary가 0 이하인 고객 수

41


In [14]:
rfm = rfm.dropna()                  # R, F가 빈 61명 제거 (dropna = 빈칸 있는 줄 지우기)
print(len(rfm))
rfm = rfm[rfm['Monetary'] > 0]      # 순구매액이 0 이하인 고객 제거
print(len(rfm))

5881
5840


In [15]:
rfm.to_csv('/content/drive/MyDrive/marketing-portfolio/data/rfm.csv')

RFM 분석 대상에서 제외: ① 기간 내 취소 기록만 있는 고객(R, F 계산 불가) ② 구매 후 취소로 순구매액이 0 이하인 고객. 취소 이력이 있어도 순구매액이 플러스인 고객은 포함. =결국 머니터리가 음수인 고객을 제외하고 분석

In [16]:
# R: 최근에 산 고객이 좋으니까, 숫자가 작을수록 높은 점수 (그래서 5,4,3,2,1 순서)
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# M: 많이 쓴 고객이 좋으니까, 숫자가 클수록 높은 점수
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

In [17]:
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm.head()   # 맨 오른쪽에 R, F, M 점수 열이 생겼는지 확인

,Recency,Frequency,Monetary,R,M,F
Customer ID,,,,,,
12347.0,2.0,8.0,5633.32,5,5,4
12348.0,75.0,5.0,2019.40,3,4,4
12349.0,19.0,4.0,4404.54,5,5,3
12350.0,310.0,1.0,334.40,2,2,1
12351.0,375.0,1.0,300.93,2,2,1


In [18]:
def segment(row):
    r = row['R']
    f = row['F']
    if r >= 4 and f >= 4:
        return 'Champions' #최근까지도 단골
    elif r >= 3 and f >= 3:
        return 'Loyal' #꾸준히 단골
    elif r >= 4 and f <= 2:
        return 'New' #최근에 새로 들어온 고객
    elif r <=2 and f >=3:
        return 'At Risk' #예전엔 자주 샀지만 최근에 안 산 고객
    elif r <= 2 and f <= 2:
        return 'Hibernating' #가끔 사고 떠난 고객
    else:
        return 'Need Attention'

rfm['Segment'] = rfm.apply(segment, axis=1)   # 고객 한 줄씩 segment 함수에 넣어서 그룹 이름 붙이기

In [19]:
print(rfm['Segment'].value_counts())                            # 그룹별 고객 수
rfm.groupby('Segment')[['Recency', 'Frequency', 'Monetary']].mean()   # 그룹별 평균

Segment
Hibernating       1518
Champions         1478
Loyal             1211
At Risk            815
New                446
Need Attention     372
Name: count, dtype: int64


,Recency,Frequency,Monetary
Segment,,,
At Risk,365.623313,4.996319,1749.197315
Champions,20.327470,15.690798,8013.893062
Hibernating,457.213439,1.260211,414.346352
Loyal,77.982659,5.417836,1997.938180
Need Attention,105.607527,1.368280,532.956352
New,28.087444,1.466368,505.740359


In [25]:
summary = rfm.groupby('Segment').agg(
    customers=('Monetary', 'count'),   # 그룹별 고객 수
    revenue=('Monetary', 'sum')        # 그룹별 매출 합계
)
summary['customer_%'] = (summary['customers'] / summary['customers'].sum() * 100).round(1)
summary['revenue_%'] = (summary['revenue'] / summary['revenue'].sum() * 100).round(1)
summary['revenue'] = summary['revenue'].round(0).astype(int)
summary.sort_values('revenue_%', ascending=False)

,customers,revenue,customer_%,revenue_%
Segment,,,,
Champions,1478,11844534,25.3,70.7
Loyal,1211,2419503,20.7,14.5
At Risk,815,1425596,14.0,8.5
Hibernating,1518,628978,26.0,3.8
New,446,225560,7.6,1.3
Need Attention,372,198260,6.4,1.2


## Key Observations

**Q1. Which segment has the most customers?**
Hibernating is the largest segment (1518 customers, 26% of all customers). Most customers buy only once or twice and do not come back, which suggests that retention is a bigger opportunity than acquisition.

**Q2. How much more do Champions spend than other segments?**
On average, Champions spend about 4 to 20 times more than the other segments. They make up 25.3% of customers but generate 70.7% of total revenue.

**Q3. How many customers are At Risk, and how long has it been since their last purchase?**
815 customers are At Risk. On average, about one year (365.6 days) has passed since their last purchase.
*Hypothesis:* Many customers are wholesalers buying seasonal gift items, so some of these customers may be annual buyers (e.g., before Christmas) rather than lost customers. Timing a win-back campaign before the holiday season may work better than a generic discount.